# Cuaderno 3 — Interpolación de Lagrange (Integrante B)
**Workshop 2 · Algoritmos Numéricos · Ingeniería Civil · Universidad del Quindío**
**Ejercicios enlazados:** Ej. 8 (3 nodos, consulta interior x=2), 9 (4 nodos), 10 (2 nodos, x=3), 11 (bases y δ de Kronecker) y 7 (2 nodos, adicional). **Módulos:** `interpolacion.py`, `datos.py`, `metricas.py`. Ejecutar con *Kernel → Restart & Run All*.

## 1. Teoría
$$P(x)=\sum_{i=0}^{n}y_iL_i(x),\qquad L_i(x)=\prod_{k\neq i}\frac{x-x_k}{x_i-x_k},\qquad L_i(x_j)=\delta_{ij}.$$
Las $L_i$ son las **ponderaciones** de cada dato; en un nodo $x_j$ solo pesa $y_j$ (reproducción de nodos). Con $n+1$ nodos el polinomio es único (grado $\le n$), así que cualquier otra forma (Newton, baricéntrica, `numpy.polyfit`) debe dar el mismo resultado.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P
from scipy.interpolate import BarycentricInterpolator, lagrange as sp_lagrange, CubicSpline
from interpolacion import *
from datos import cargar, control_calidad, particion_comun, comparar, fraccion_regular, codigos_calificacion
import metricas
pd.options.display.float_format = '{:.5f}'.format

## 2. Ejercicios manuales vs. código
Para cada caso se compara el polinomio manual con el obtenido exactamente por `np.polyfit` (grado $n$) y se evalúa con `lagrange_eval`.

In [ ]:
casos = {
 "Ej. 7  (2 nodos)":  ([2, 6], [5, 17], [-1, 3], [4]),                         # P=3x-1
 "Ej. 10 (2 nodos, x=3)": ([2, 5], [5, 11], [1, 2], [3]),                      # P=2x+1 -> 7
 "Ej. 8  (3 nodos, x=2)": ([1, 3, 4], [2, 6, 24], [16, -58/3, 16/3], [2]),       # 16 - 58/3 x + 16/3 x^2
 "Ej. 9  (4 nodos)":  ([-1, 0, 1, 2], [4, 1, 2, 13], [1, -2, 2, 1], [0.5]),    # x^3+2x^2-2x+1 (coef asc.)
}
filas = []
for nom, (x, y, ca, xq) in casos.items():
    x, y = np.array(x, float), np.array(y, float)
    cfit = np.polyfit(x, y, len(x) - 1)[::-1]                                    # ascendente
    manual = P.polyval(xq, ca); lag = lagrange_eval(x, y, xq)
    filas.append({"caso": nom, "coef. manual": np.round(ca, 4).tolist(), "coef. polyfit": np.round(cfit, 4).tolist(),
                  "P(xq) manual": manual[0], "P(xq) Lagrange": lag[0], "|dif|": abs(manual[0] - lag[0]),
                  "residual nodos": np.abs(lagrange_eval(x, y, x) - y).max()})
pd.options.display.float_format = '{:.6f}'.format
display(pd.DataFrame(filas).set_index("caso"))
print("Ej. 8: P(2) =", lagrange_eval([1,3,4],[2,6,24],[2])[0], " (manual -4/3 =", -4/3, ")")

**Ej. 8 — consulta interior:** $x=2\in[1,3]$ da $P_2(2)=-4/3$, valor **negativo** aunque todos los datos son positivos: el polinomio global no garantiza positividad. Es una decisión relevante para explicar (guía: *consulta interior y adaptación justificada*). **Adaptación propuesta:** reemplazar $(4,24)$ por un valor consistente con una tendencia suave y comparar; documentar qué cambia en $P_2(2)$.

## 3. Ejercicio 9/11 — bases, ponderaciones y δ de Kronecker
Nodos $(0,1),(1,2),(2,5),(3,10)$ (datos de $y=x^2+1$).

In [ ]:
xk = np.array([0., 1, 2, 3]); yk = xk**2 + 1
K = np.array([[bases_lagrange(xk, i, xj) for xj in xk] for i in range(4)])
display(pd.DataFrame(K, index=[f"L_{i}" for i in range(4)], columns=[f"x={x:g}" for x in xk]))    # debe ser la identidad
W = np.array([bases_lagrange(xk, i, 1.5) for i in range(4)])
display(pd.DataFrame({"L_i(1.5)": W, "y_i": yk, "y_i*L_i": yk*W}))
print("Σ L_i(1.5) =", W.sum(), "| P(1.5) =", (yk*W).sum(), "| exacto 1.5^2+1 =", 1.5**2 + 1)

## 4. Contraste con bibliotecas

In [ ]:
x = np.array([-1., 0, 1, 2]); y = np.array([4., 1, 2, 13]); xx = np.linspace(-1, 2, 50)
print("max|Lagrange - Barycentric SciPy| =", np.abs(lagrange_eval(x, y, xx) - BarycentricInterpolator(x, y)(xx)).max())
print("max|Lagrange - scipy.lagrange|    =", np.abs(lagrange_eval(x, y, xx) - sp_lagrange(x, y)(xx)).max())

## 5. Datos reales: Lagrange local (2, 3, 4 nodos) vs. global
Se usa la **misma partición** que en los Cuadernos 4 y 5 (ver ahí la documentación y el control de calidad de la base USGS).

In [ ]:
df, es_real = cargar(n_recorte=36)
nodos, val = particion_comun(len(df), 4, 2); R = comparar(df, nodos, val)
tn, yn = R["t"][nodos], R["y"][nodos]
filas = []
for k in (2, 3, 4):
    pred = [sum(yn[w][j]*bases_lagrange(tn[w], j, tv) for j in range(k)) for tv in R["t_val"] for w in [ventana_nodos(tn, tv, k)]]
    r = metricas.resumen(pred, R["y_val"], "ft³/s"); filas.append({"método": f"Lagrange local {k} nodos", "MAE": r["MAE"], "RMSE": r["RMSE"], "E_max": r["E_max"]})
g = R["metricas"].loc["Lagrange"]; filas.append({"método": f"Lagrange global ({len(nodos)} nodos)", "MAE": g.MAE, "RMSE": g.RMSE, "E_max": g.E_max})
pd.options.display.float_format = '{:.3f}'.format
display(pd.DataFrame(filas).set_index("método").rename(columns={"MAE": "MAE [ft³/s]", "RMSE": "RMSE [ft³/s]", "E_max": "E_máx [ft³/s]"}))

## 6. Cierre
**Comprobé:** coincidencia manual–código, matriz $L_i(x_j)=I$, $\sum L_i=1$ y acuerdo con SciPy. **Límites:** el polinomio global con muchos nodos equiespaciados oscila (Runge) y puede tener un E_máx mucho mayor que el de la interpolación local (verificar con las cifras); la reproducción de nodos no mide el error entre nodos; la discrepancia frente a reservadas no es una cota del error físico.

**Registro de IA:** ver `registro_IA.md` y `history.md`. *(Completar con fecha, herramienta/modelo tal como se muestra, prompt exacto, qué se aceptó/modificó/descartó y la comprobación independiente; o declarar «Sin uso de IA».)*